# 02 - Configure

This notebook performs lightweight post-deployment checks and prepares defaults used by the staged test notebooks.

Pre-requisite: run `01_deploy_infra.ipynb` first so `env/.env` exists.

---

## SDK guidance

- Use `azure-ai-inference` for GPT-4o inference calls.
- Use `azure-cognitiveservices-speech` for speech scenarios.
- Keep this notebook small and focused on validating configuration, not running full tests.

---

## Step 1 – Load environment variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

# Read the values written by 01_deploy_infra.ipynb
# endpoint    = os.environ["AZURE_AI_ENDPOINT"]
# auth_mode   = os.environ.get("AZURE_AUTH_MODE", "aad")
# deployment  = os.environ.get("AZURE_AI_DEPLOYMENT", "chat")

print("Environment loaded")

## Step 2 – Create SDK client(s)

For model inference use `azure-ai-inference`. For other services use the respective Azure SDK.

In [ ]:
# TODO: initialise the relevant SDK client(s), e.g.:
#
# from azure.ai.inference import ChatCompletionsClient
# from azure.identity import DefaultAzureCredential
#
# client = ChatCompletionsClient(
#     endpoint=endpoint,
#     credential=DefaultAzureCredential(),
# )
#
# For AI Search (RAG demos), prefer managed identity / Entra auth where possible.

print("TODO: create SDK client(s)")

## Step 3 – Configure

Perform the configuration steps required for your demo. Add as many cells as needed.
Label any `[PREVIEW]` features clearly in the markdown above each cell.

In [ ]:
from pathlib import Path

required_keys = [
    "AZURE_RESOURCE_GROUP",
    "AZURE_LOCATION",
    "AZURE_ACCOUNT_NAME",
    "AZURE_AI_ENDPOINT",
    "AZURE_AUTH_MODE",
    "AZURE_AI_DEPLOYMENT",
    "AZURE_SPEECH_VOICE",
    "LOCAL_IMAGE_PATH",
    "LOCAL_AUDIO_PATH",
]

missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing required env keys: {missing}")

print("Required env keys are present")
print(f"AI endpoint: {os.environ['AZURE_AI_ENDPOINT']}")
print(f"Auth mode  : {os.environ['AZURE_AUTH_MODE']}")
print(f"Deployment : {os.environ['AZURE_AI_DEPLOYMENT']}")
print(f"Voice      : {os.environ['AZURE_SPEECH_VOICE']}")

# Ensure outputs and data directories exist for downstream notebooks.
Path("../outputs").mkdir(exist_ok=True)
Path("../data").mkdir(exist_ok=True)

In [ ]:
from pathlib import Path

example_image = Path(os.environ["LOCAL_IMAGE_PATH"]).expanduser()
example_audio = Path(os.environ["LOCAL_AUDIO_PATH"]).expanduser()

print(f"Image path configured: {example_image}")
print(f"Audio path configured: {example_audio}")

if not example_image.exists():
    print("Note: local image file does not exist yet. Add a sample before running 04_image_test.ipynb.")
if not example_audio.exists():
    print("Note: local audio file does not exist yet. Add a sample before running 05_audio_text.ipynb.")

## Step 4 – Verify configuration

In [ ]:
import requests
from azure.identity import AzureCliCredential

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()

if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = AzureCliCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "AzureCliCredential"
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

url = f"{endpoint}/openai/deployments/{deployment}/chat/completions?api-version=2024-10-21"
payload = {
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Respond with: configuration check passed."},
    ],
    "max_completion_tokens": 32,
}

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
response = requests.post(url, headers=headers, json=payload, timeout=45)
if response.status_code >= 400:
    raise RuntimeError(f"Model check failed: {response.status_code} {response.text}")

content = response.json()["choices"][0]["message"]["content"]
print("Model connectivity check succeeded")
print(f"Credential source: {credential_source}")
print(content)

---

## Configuration complete

Move to **03_text_test.ipynb** to run the first functional checkpoint (text-only).